# Programming Assignment: Building a Naive Bayes Classifier from Scratch

Welcome to the assignment for Naive Bayes classification.

In earlier examples, you've worked with black-box machine learning models from scikit-learn. While these are powerful and convenient, understanding how they work under the hood is essential for becoming a skilled machine learning practitioner. Naive Bayes is one of the most elegant and interpretable algorithms in machine learning, based on probability theory and Bayes' theorem.

In this assignment, you'll implement Naive Bayes classifiers from scratch using only NumPy. You'll work with both continuous features (Gaussian Naive Bayes) and discrete features (Multinomial Naive Bayes), applying them to real-world classification problems including spam detection.

Naive Bayes makes the "naive" assumption that features are conditionally independent given the class label. Despite this simplification, it performs remarkably well in many practical applications, especially with text classification, medical diagnosis, and spam filtering.

**What You will do in this Assignment**

* Understand the mathematical foundation of Naive Bayes using Bayes' theorem.
* Implement Gaussian Naive Bayes for continuous features from scratch.
* Calculate likelihood probabilities using the Gaussian distribution.
* Apply Bayes' theorem to make predictions with probability outputs.
* Implement probability calibration to improve prediction confidence.
* Build Multinomial Naive Bayes for text classification and discrete features.
* Compare Naive Bayes with Logistic Regression to understand when each excels.
* Create a complete spam classifier with interpretable probability scores.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - Introduction to Naive Bayes](#1)
    - [1.1 - Understanding Bayes' Theorem](#1-1)
    - [1.2 - The Naive Assumption](#1-2)
- [2 - Gaussian Naive Bayes](#2)
    - **[Exercise 1 - calculate_statistics](#ex-1)**
    - **[Exercise 2 - gaussian_likelihood](#ex-2)**
    - **[Exercise 3 - predict_with_probabilities](#ex-3)**
- [3 - Model Application and Analysis](#3)
    - [3.1 - Loading and Preparing Data](#3-1)
    - **[Exercise 4 - analyze_predictions](#ex-4)**
- [4 - Probability Calibration](#4)
    - **[Exercise 5 - calibrate_probabilities](#ex-5)**
- [5 - Multinomial Naive Bayes](#5)
    - **[Exercise 6 - multinomial_naive_bayes](#ex-6)**
- [6 - Comparative Analysis](#6)
    - **[Exercise 7 - compare_classifiers](#ex-7)**
- [7 - Spam Classification](#7)
    - **[Exercise 8 - build_spam_classifier](#ex-8)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import unittests

from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

<a name='1'></a>
## 1 - Introduction to Naive Bayes

Naive Bayes is a probabilistic classifier based on applying Bayes' theorem with the "naive" assumption of conditional independence between features. Despite its simplicity, it's highly effective for many real-world applications.

<a name='1-1'></a>
### 1.1 - Understanding Bayes' Theorem

Bayes' theorem describes the probability of an event based on prior knowledge of conditions related to the event:

$$P(y|X) = \frac{P(X|y) \cdot P(y)}{P(X)}$$

Where:
- $P(y|X)$ is the **posterior probability**: the probability of class $y$ given features $X$
- $P(X|y)$ is the **likelihood**: the probability of observing features $X$ given class $y$
- $P(y)$ is the **prior probability**: the probability of class $y$ before seeing the data
- $P(X)$ is the **evidence**: the probability of observing features $X$

Since we're comparing probabilities across classes for the same instance, $P(X)$ is constant and can be ignored for classification.

<a name='1-2'></a>
### 1.2 - The Naive Assumption

The "naive" assumption states that all features are conditionally independent given the class:

$$P(X|y) = P(x_1, x_2, ..., x_n|y) = \prod_{i=1}^{n} P(x_i|y)$$

This simplifies computation dramatically, as we only need to calculate individual feature probabilities rather than joint probabilities of all feature combinations.

The final prediction is made by choosing the class with the highest posterior probability:

$$\hat{y} = \arg\max_{y} P(y) \prod_{i=1}^{n} P(x_i|y)$$

<a name='2'></a>
## 2 - Gaussian Naive Bayes

When dealing with continuous features, we assume each feature follows a Gaussian (normal) distribution within each class. The probability density function for a Gaussian distribution is:

$$P(x_i|y) = \frac{1}{\sqrt{2\pi\sigma_y^2}} \exp\left(-\frac{(x_i - \mu_y)^2}{2\sigma_y^2}\right)$$

Where $\mu_y$ and $\sigma_y^2$ are the mean and variance of feature $x_i$ for class $y$.

<a name='ex-1'></a>
#### **Exercise 1 - `calculate_statistics`**

**Your Task:**

Implement the `calculate_statistics` function that computes the mean, variance, and prior probabilities for each class in the training data. These statistics form the foundation of Gaussian Naive Bayes.

You need to implement the following:

* **Calculate class prior probabilities**:
    * Count the number of samples in each class.
    * Divide by the total number of samples to get $P(y)$.

* **Calculate mean for each feature per class**:
    * For each class, compute the mean of each feature using samples belonging to that class.
    * Store results in a dictionary with class labels as keys.

* **Calculate variance for each feature per class**:
    * For each class, compute the variance of each feature.
    * Add a small epsilon ($10^{-9}$) to prevent division by zero.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help, here's a more detailed guide:

**For class prior probabilities:**
* Use `np.unique(y, return_counts=True)` to get unique classes and their counts.
* Divide counts by `len(y)` to get probabilities.
* Store in a dictionary: `{class: probability}`.

**For mean calculation:**
* Iterate through each class.
* Use boolean indexing: `X[y == class_label]` to get samples of that class.
* Use `np.mean(X_class, axis=0)` to compute mean for each feature.

**For variance calculation:**
* Similar to mean, use `X[y == class_label]` for each class.
* Use `np.var(X_class, axis=0)` to compute variance.
* Add epsilon: `variance + 1e-9`.

</details>

In [ ]:
# GRADED FUNCTION: calculate_statistics
def calculate_statistics(X, y):
    """
    Calculate mean, variance, and prior probabilities for each class.
    
    Args:
        X: numpy array of shape (n_samples, n_features) - Training features
        y: numpy array of shape (n_samples,) - Training labels
    
    Returns:
        means: dict mapping class labels to mean vectors of shape (n_features,)
        variances: dict mapping class labels to variance vectors of shape (n_features,)
        priors: dict mapping class labels to prior probabilities (float)
    """
    means = {}
    variances = {}
    priors = {}
    
    ### START CODE HERE ###
    
    # Get unique classes and their counts
    
    # Calculate prior probabilities for each class
    
    # Calculate mean and variance for each class
    
    ### END CODE HERE ###
    
    return means, variances, priors

In [ ]:
# Test with a simple dataset
X_test = np.array([[1, 2], [2, 3], [3, 4], [6, 7], [7, 8], [8, 9]])
y_test = np.array([0, 0, 0, 1, 1, 1])

means, variances, priors = calculate_statistics(X_test, y_test)

print("Means per class:")
for class_label, mean in means.items():
    print(f"  Class {class_label}: {mean}")

print("\nVariances per class:")
for class_label, var in variances.items():
    print(f"  Class {class_label}: {var}")

print("\nPrior probabilities:")
for class_label, prior in priors.items():
    print(f"  Class {class_label}: {prior:.4f}")

##### **Expected Output**
```
Means per class:
  Class 0: [2. 3.]
  Class 1: [7. 8.]

Variances per class:
  Class 0: [0.66666667 0.66666667]
  Class 1: [0.66666667 0.66666667]

Prior probabilities:
  Class 0: 0.5000
  Class 1: 0.5000
```

In [ ]:
unittests.exercise_1(calculate_statistics)

<a name='ex-2'></a>
#### **Exercise 2 - `gaussian_likelihood`**

**Your Task:**

Implement the `gaussian_likelihood` function that calculates the probability of observing a feature value given a class, assuming a Gaussian distribution.

You need to implement:

* **Calculate the Gaussian probability density**:
    * Use the formula: $P(x|y) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$
    * Compute the exponent term: $-\frac{(x - \mu)^2}{2\sigma^2}$
    * Compute the coefficient: $\frac{1}{\sqrt{2\pi\sigma^2}}$
    * Multiply them together to get the final probability

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For the Gaussian formula:**
* Calculate exponent: `exponent = -((x - mean) ** 2) / (2 * variance)`
* Calculate coefficient: `coef = 1 / np.sqrt(2 * np.pi * variance)`
* Final probability: `probability = coef * np.exp(exponent)`
* Use `np.prod()` to multiply probabilities across all features

</details>

In [ ]:
# GRADED FUNCTION: gaussian_likelihood
def gaussian_likelihood(x, mean, variance):
    """
    Calculate the Gaussian probability density function for a given observation.
    
    Args:
        x: numpy array of shape (n_features,) - Feature values for a single sample
        mean: numpy array of shape (n_features,) - Mean values for each feature
        variance: numpy array of shape (n_features,) - Variance values for each feature
    
    Returns:
        likelihood: float - The joint probability of all features (product of individual probabilities)
    """
    ### START CODE HERE ###
    
    # Calculate the exponent term
    
    # Calculate the coefficient
    
    # Calculate probability density for each feature
    
    # Return the product of all feature probabilities
    
    ### END CODE HERE ###
    
    return likelihood

In [ ]:
# Test the gaussian_likelihood function
x_sample = np.array([2.0, 3.0])
mean_test = np.array([2.0, 3.0])
var_test = np.array([1.0, 1.0])

likelihood = gaussian_likelihood(x_sample, mean_test, var_test)
print(f"Likelihood when x equals mean: {likelihood:.6f}")

x_sample2 = np.array([3.0, 4.0])
likelihood2 = gaussian_likelihood(x_sample2, mean_test, var_test)
print(f"Likelihood when x differs from mean: {likelihood2:.6f}")

##### **Expected Output**
```
Likelihood when x equals mean: 0.159155
Likelihood when x differs from mean: 0.058550
```

In [ ]:
unittests.exercise_2(gaussian_likelihood)

<a name='ex-3'></a>
#### **Exercise 3 - `predict_with_probabilities`**

**Your Task:**

Implement the `predict_with_probabilities` function that uses Bayes' theorem to predict class labels with probability scores for each sample in the dataset.

You need to implement:

* **Calculate posterior probabilities for each class**:
    * For each sample, compute the posterior probability for each class.
    * Use the formula: $P(y|X) \propto P(y) \times P(X|y)$
    * The posterior is the prior probability multiplied by the likelihood.

* **Normalize probabilities**:
    * Sum all posterior probabilities across classes.
    * Divide each posterior by this sum to get normalized probabilities.

* **Make predictions**:
    * Choose the class with the highest posterior probability.
    * Return both predictions and probability distributions.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For probability calculation:**
* Initialize a dictionary to store posteriors for each class.
* Loop through each class and calculate: `prior * gaussian_likelihood(x, mean, variance)`.
* Store results in the dictionary.

**For normalization:**
* Sum all posterior values: `total = sum(posteriors.values())`.
* Divide each posterior by total: `{class: prob/total for class, prob in posteriors.items()}`.

**For prediction:**
* Use `max(posteriors, key=posteriors.get)` to find class with highest probability.
* Store predictions in a list and probabilities in another list or array.

</details>

In [ ]:
# GRADED FUNCTION: predict_with_probabilities
def predict_with_probabilities(X, means, variances, priors):
    """
    Predict class labels and return probability distributions for each sample.
    
    Args:
        X: numpy array of shape (n_samples, n_features) - Samples to predict
        means: dict mapping class labels to mean vectors
        variances: dict mapping class labels to variance vectors
        priors: dict mapping class labels to prior probabilities
    
    Returns:
        predictions: numpy array of shape (n_samples,) - Predicted class labels
        probabilities: list of dicts - Each dict maps class labels to probabilities for that sample
    """
    predictions = []
    probabilities = []
    
    ### START CODE HERE ###
    
    # For each sample in X
    
        # Calculate posterior probability for each class
        
        # Normalize probabilities
        
        # Predict class with highest probability
        
    ### END CODE HERE ###
    
    return np.array(predictions), probabilities

In [ ]:
# Test the predict_with_probabilities function
X_test_pred = np.array([[2, 3], [7, 8], [4, 5]])

predictions, probs = predict_with_probabilities(X_test_pred, means, variances, priors)

print("Predictions:", predictions)
print("\nProbability distributions:")
for i, prob_dist in enumerate(probs):
    print(f"Sample {i}: {prob_dist}")

##### **Expected Output**
```
Predictions: [0 1 0]

Probability distributions:
Sample 0: {0: 0.8807970779778824, 1: 0.11920292202211755}
Sample 1: {0: 0.11920292202211755, 1: 0.8807970779778824}
Sample 2: {0: 0.7310585786300049, 1: 0.2689414213699951}
```

In [ ]:
unittests.exercise_3(predict_with_probabilities)

<a name='3'></a>
## 3 - Model Application and Analysis

Now that you have implemented the core Gaussian Naive Bayes algorithm, let's apply it to real-world datasets and analyze its performance.

<a name='3-1'></a>
### 3.1 - Loading and Preparing Data

We'll use the Iris dataset, a classic machine learning dataset with continuous features representing flower measurements.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"Number of classes: {len(np.unique(y_train))}")

In [ ]:
# Train the Gaussian Naive Bayes model
means_iris, variances_iris, priors_iris = calculate_statistics(X_train, y_train)

# Make predictions
y_pred, y_probs = predict_with_probabilities(X_test, means_iris, variances_iris, priors_iris)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy: {accuracy:.4f}")

# Display confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

<a name='ex-4'></a>
#### **Exercise 4 - `analyze_predictions`**

**Your Task:**

Implement the `analyze_predictions` function that analyzes the quality of predictions by examining confidence levels and identifying uncertain predictions.

You need to implement:

* **Calculate confidence scores**:
    * For each prediction, the confidence is the maximum probability among all classes.
    * Extract this from the probability distributions returned by `predict_with_probabilities`.

* **Identify high and low confidence predictions**:
    * Sort predictions by confidence level.
    * Find the top N most confident predictions.
    * Find the top N least confident (most uncertain) predictions.

* **Calculate statistics**:
    * Compute average confidence for correct and incorrect predictions.
    * Count how many predictions fall below a confidence threshold (e.g., 0.5).

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For confidence scores:**
* Extract max probability for each sample: `confidence = max(prob_dict.values())`.
* Store in a list or array.

**For high/low confidence:**
* Create list of (index, confidence) tuples.
* Sort by confidence: `sorted(items, key=lambda x: x[1])`.
* Take first N for lowest, last N for highest.

**For statistics:**
* Use boolean indexing: `confidences[predictions == y_true]` for correct predictions.
* Calculate mean: `np.mean(correct_confidences)`.
* Count low confidence: `sum(confidences < threshold)`.

</details>

In [ ]:
# GRADED FUNCTION: analyze_predictions
def analyze_predictions(y_true, y_pred, probabilities, top_n=5):
    """
    Analyze prediction quality by examining confidence levels.
    
    Args:
        y_true: numpy array of shape (n_samples,) - True labels
        y_pred: numpy array of shape (n_samples,) - Predicted labels
        probabilities: list of dicts - Probability distributions for each sample
        top_n: int - Number of top confident/uncertain predictions to return
    
    Returns:
        analysis: dict containing:
            - 'confidences': numpy array of confidence scores
            - 'avg_confidence': float - Overall average confidence
            - 'avg_confidence_correct': float - Average confidence for correct predictions
            - 'avg_confidence_incorrect': float - Average confidence for incorrect predictions
            - 'most_confident': list of (index, confidence) tuples
            - 'least_confident': list of (index, confidence) tuples
            - 'low_confidence_count': int - Count of predictions below 0.5 confidence
    """
    analysis = {}
    
    ### START CODE HERE ###
    
    # Extract confidence scores (max probability for each sample)
    
    # Calculate average confidence for correct predictions
    
    # Calculate average confidence for incorrect predictions
    
    # Find most and least confident predictions
    
    # Count predictions with confidence below 0.5
    
    ### END CODE HERE ###
    
    return analysis

In [ ]:
# Test the analyze_predictions function
analysis_results = analyze_predictions(y_test, y_pred, y_probs, top_n=3)

print(f"Average confidence (correct): {analysis_results['avg_confidence_correct']:.4f}")
print(f"Average confidence (incorrect): {analysis_results['avg_confidence_incorrect']:.4f}")
print(f"\nLow confidence predictions (< 0.5): {analysis_results['low_confidence_count']}")

print("\nMost confident predictions:")
for idx, conf in analysis_results['most_confident']:
    print(f"  Sample {idx}: confidence = {conf:.4f}, true = {y_test[idx]}, pred = {y_pred[idx]}")

print("\nLeast confident predictions:")
for idx, conf in analysis_results['least_confident']:
    print(f"  Sample {idx}: confidence = {conf:.4f}, true = {y_test[idx]}, pred = {y_pred[idx]}")

##### **Expected Output** (values may vary slightly)
```
Average confidence (correct): 0.9700
Average confidence (incorrect): 0.7500

Low confidence predictions (< 0.5): 0

Most confident predictions:
  Sample X: confidence = 1.0000, true = Y, pred = Y
  ...

Least confident predictions:
  Sample X: confidence = 0.XXXX, true = Y, pred = Z
  ...
```

In [ ]:
unittests.exercise_4(analyze_predictions)

<a name='4'></a>
## 4 - Probability Calibration

While Naive Bayes provides probability estimates, these probabilities are often not well-calibrated. Calibration adjusts the predicted probabilities to better reflect the true likelihood of the prediction being correct.

A well-calibrated classifier should have the property that among all predictions with confidence 0.8, approximately 80% are correct.

<a name='ex-5'></a>
#### **Exercise 5 - `calibrate_probabilities`**

**Your Task:**

Implement the `calibrate_probabilities` function using Platt scaling (logistic calibration). This technique fits a logistic regression model on the predicted probabilities to produce calibrated probabilities.

You need to implement:

* **Extract confidence scores**:
    * Get the maximum probability for each prediction.
    * These will be the features for the calibration model.

* **Fit a simple logistic calibration**:
    * Create binary labels: 1 if prediction is correct, 0 otherwise.
    * Fit a logistic transformation: $p_{calibrated} = \frac{1}{1 + e^{-(a \cdot p + b)}}$
    * You can use a simplified approach or sklearn's LogisticRegression.

* **Apply calibration**:
    * Transform the confidence scores using the fitted parameters.
    * Return calibrated probabilities.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For extracting scores:**
* Loop through probabilities: `scores = [max(p.values()) for p in probabilities]`.

**For binary labels:**
* Create correctness labels: `correct = (y_pred == y_true).astype(int)`.

**For calibration (simplified approach):**
* Use sklearn's LogisticRegression: `from sklearn.linear_model import LogisticRegression`.
* Fit on confidence scores: `model.fit(scores.reshape(-1, 1), correct)`.
* Predict calibrated probabilities: `calibrated = model.predict_proba(scores.reshape(-1, 1))[:, 1]`.

**Alternative: Manual implementation:**
* Calculate mean probability and mean correctness by bins.
* Apply isotonic regression or histogram binning.

</details>

In [ ]:
# GRADED FUNCTION: calibrate_probabilities
def calibrate_probabilities(y_true, y_pred, probabilities):
    """
    Calibrate predicted probabilities using Platt scaling.
    
    Args:
        y_true: numpy array of shape (n_samples,) - True labels
        y_pred: numpy array of shape (n_samples,) - Predicted labels
        probabilities: list of dicts OR numpy array of confidence scores

    Returns:
        calibrated_probs: numpy array of shape (n_samples,) - Calibrated confidence scores
        calibration_model: fitted calibration model (for applying to new data)
    """
    from sklearn.linear_model import LogisticRegression
    
    ### START CODE HERE ###
    
    # Extract confidence scores (max probability for each sample)
    
    # Create binary correctness labels
    
    # Fit logistic calibration model
    
    # Apply calibration to get calibrated probabilities
    
    ### END CODE HERE ###
    
    return calibrated_probs, calibration_model

In [ ]:
# Test the calibrate_probabilities function
calibrated_probs, calib_model = calibrate_probabilities(y_test, y_pred, y_probs)

print("Original vs Calibrated Probabilities (first 10 samples):")
print("Original | Calibrated | Correct?")
print("-" * 40)
for i in range(min(10, len(y_pred))):
    original_conf = max(y_probs[i].values())
    correct = "✓" if y_pred[i] == y_test[i] else "✗"
    print(f"{original_conf:.4f}   | {calibrated_probs[i]:.4f}     | {correct}")

##### **Expected Output** (values will vary)
```
Original vs Calibrated Probabilities (first 10 samples):
Original | Calibrated | Correct?
----------------------------------------
0.9999   | 0.9856     | ✓
0.9856   | 0.9745     | ✓
...
```

In [ ]:
unittests.exercise_5(calibrate_probabilities)

<a name='5'></a>
## 5 - Multinomial Naive Bayes

So far, we've worked with continuous features using Gaussian Naive Bayes. Now let's implement **Multinomial Naive Bayes**, which is designed for discrete features like word counts in text classification.

In Multinomial Naive Bayes, features represent counts (e.g., word frequencies), and we model the likelihood using the multinomial distribution:

$$P(x_i|y) = \frac{N_{yi} + \alpha}{N_y + \alpha n}$$

Where:
- $N_{yi}$ is the count of feature $i$ in class $y$
- $N_y$ is the total count of all features in class $y$
- $\alpha$ is the smoothing parameter (typically 1, called Laplace smoothing)
- $n$ is the number of features

<a name='ex-6'></a>
#### **Exercise 6 - `multinomial_naive_bayes`**

**Your Task:**

Implement a complete Multinomial Naive Bayes classifier that can fit on training data and predict on test data.

You need to implement:

* **Fit method - Calculate feature probabilities**:
    * For each class, count occurrences of each feature value.
    * Apply Laplace smoothing to avoid zero probabilities.
    * Calculate prior probabilities for each class.

* **Predict method - Make predictions**:
    * For each test sample, calculate log probabilities for each class.
    * Use log probabilities to avoid numerical underflow: $\log P(y|X) = \log P(y) + \sum_i \log P(x_i|y)$
    * Choose the class with the highest log probability.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For fit method:**
* Calculate class priors: `priors[c] = (y == c).sum() / len(y)`.
* For each class, sum feature counts: `feature_counts = X[y == c].sum(axis=0)`.
* Apply Laplace smoothing: `(feature_counts + alpha) / (feature_counts.sum() + alpha * n_features)`.
* Store in a dictionary: `self.feature_log_probs[class] = np.log(probabilities)`.

**For predict method:**
* Calculate log likelihood: `log_likelihood = X @ self.feature_log_probs[class]`.
* Add log prior: `log_posterior = log_prior + log_likelihood`.
* Choose class with max log posterior: `np.argmax(log_posteriors, axis=1)`.

</details>

In [ ]:
# GRADED FUNCTION: MultinomialNB class
class MultinomialNB:
    """
    Multinomial Naive Bayes classifier for discrete features.
    """
    def __init__(self, alpha=1.0):
        """
        Initialize the classifier.
        
        Args:
            alpha: float - Smoothing parameter (Laplace smoothing)
        """
        self.alpha = alpha
        self.classes = None
        self.feature_log_probs = {}
        self.class_log_priors = {}
    
    def fit(self, X, y):
        """
        Fit the Multinomial Naive Bayes model.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Training features (counts)
            y: numpy array of shape (n_samples,) - Training labels
        """
        ### START CODE HERE ###
        
        # Get unique classes
        
        # Calculate class priors
        
        # For each class, calculate feature probabilities with Laplace smoothing
        
        ### END CODE HERE ###
        
        return self
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Test features
        
        Returns:
            predictions: numpy array of shape (n_samples,) - Predicted labels
        """
        ### START CODE HERE ###
        
        # Calculate log posterior for each class
        
        # Choose class with highest log posterior
        
        ### END CODE HERE ###
        
        return predictions
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X.
        
        Args:
            X: numpy array of shape (n_samples, n_features) - Test features
        
        Returns:
            probabilities: numpy array of shape (n_samples, n_classes) - Class probabilities
        """
        ### START CODE HERE ###
        
        # Calculate log posteriors
        
        # Convert to probabilities using softmax
        
        ### END CODE HERE ###
        
        return probabilities

In [ ]:
# Test with a simple text-like dataset
# Create a toy dataset with word counts
X_text = np.array([
    [3, 0, 1, 0, 0, 5],  # Document 1 (class 0)
    [2, 0, 0, 1, 0, 4],  # Document 2 (class 0)
    [0, 4, 0, 0, 3, 1],  # Document 3 (class 1)
    [0, 3, 0, 1, 2, 0],  # Document 4 (class 1)
    [4, 0, 2, 0, 0, 6],  # Document 5 (class 0)
    [0, 5, 0, 0, 4, 0],  # Document 6 (class 1)
])
y_text = np.array([0, 0, 1, 1, 0, 1])

# Fit the model
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_text, y_text)

# Make predictions
X_test_text = np.array([
    [3, 0, 1, 0, 0, 4],  # Similar to class 0
    [0, 4, 0, 0, 3, 0],  # Similar to class 1
])

predictions = mnb.predict(X_test_text)
probabilities = mnb.predict_proba(X_test_text)

print("Predictions:", predictions)
print("\nProbabilities:")
print(probabilities)

##### **Expected Output**
```
Predictions: [0 1]

Probabilities:
[[0.99... 0.00...]
 [0.00... 0.99...]]
```

In [ ]:
unittests.exercise_6(MultinomialNB)

<a name='6'></a>
## 6 - Comparative Analysis

Now let's compare Naive Bayes with Logistic Regression to understand the strengths and weaknesses of each algorithm.

<a name='ex-7'></a>
#### **Exercise 7 - `compare_classifiers`**

**Your Task:**

Implement the `compare_classifiers` function that trains both Naive Bayes and Logistic Regression on the same dataset and compares their performance across multiple metrics.

You need to implement:

* **Train both models**:
    * Use your Gaussian Naive Bayes implementation.
    * Use sklearn's LogisticRegression for comparison.

* **Make predictions**:
    * Get predictions and probabilities from both models.

* **Calculate metrics**:
    * Accuracy for both models.
    * Training time for both models.
    * Average prediction confidence for both models.

* **Return comparison dictionary**:
    * Include all metrics for easy comparison.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For Naive Bayes:**
* Use functions you've already implemented: `calculate_statistics` and `predict_with_probabilities`.
* Time the training: `import time; start = time.time(); ...; duration = time.time() - start`.

**For Logistic Regression:**
* Import: `from sklearn.linear_model import LogisticRegression`.
* Fit: `model = LogisticRegression().fit(X_train, y_train)`.
* Predict: `y_pred = model.predict(X_test)` and `y_proba = model.predict_proba(X_test)`.

**For metrics:**
* Accuracy: `accuracy_score(y_test, y_pred)`.
* Confidence: `np.max(probabilities, axis=1).mean()` for sklearn output.
* For NB: `np.array([max(p.values()) for p in nb_probs]).mean()`.

</details>

In [ ]:
# GRADED FUNCTION: compare_classifiers
def compare_classifiers(X_train, X_test, y_train, y_test):
    """
    Compare Gaussian Naive Bayes with Logistic Regression.
    
    Args:
        X_train: numpy array - Training features
        X_test: numpy array - Test features
        y_train: numpy array - Training labels
        y_test: numpy array - Test labels
    
    Returns:
        comparison: dict containing:
            - 'nb_accuracy': float
            - 'lr_accuracy': float
            - 'nb_train_time': float (in seconds)
            - 'lr_train_time': float (in seconds)
            - 'nb_avg_confidence': float
            - 'lr_avg_confidence': float
            - 'nb_predictions': numpy array
            - 'lr_predictions': numpy array
    """
    import time
    from sklearn.linear_model import LogisticRegression
    
    comparison = {}
    
    ### START CODE HERE ###
    
    # Train Naive Bayes
    
    # Make NB predictions
    
    # Calculate NB metrics
    
    # Train Logistic Regression
    
    # Make LR predictions
    
    # Calculate LR metrics
    
    ### END CODE HERE ###
    
    return comparison

In [ ]:
# Compare classifiers on Iris dataset
comparison_results = compare_classifiers(X_train, X_test, y_train, y_test)

print("=" * 60)
print("CLASSIFIER COMPARISON")
print("=" * 60)
print(f"\n{'Metric':<30} {'Naive Bayes':<15} {'Logistic Regression':<15}")
print("-" * 60)
print(f"{'Accuracy':<30} {comparison_results['nb_accuracy']:<15.4f} {comparison_results['lr_accuracy']:<15.4f}")
print(f"{'Training Time (s)':<30} {comparison_results['nb_train_time']:<15.6f} {comparison_results['lr_train_time']:<15.6f}")
print(f"{'Average Confidence':<30} {comparison_results['nb_avg_confidence']:<15.4f} {comparison_results['lr_avg_confidence']:<15.4f}")
print("=" * 60)

# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_nb = confusion_matrix(y_test, comparison_results['nb_predictions'])
cm_lr = confusion_matrix(y_test, comparison_results['lr_predictions'])

axes[0].imshow(cm_nb, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set_title('Naive Bayes Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

axes[1].imshow(cm_lr, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set_title('Logistic Regression Confusion Matrix')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.show()

##### **Expected Output** (values will vary)
```
============================================================
CLASSIFIER COMPARISON
============================================================

Metric                         Naive Bayes     Logistic Regression
------------------------------------------------------------
Accuracy                       0.9778          0.9778
Training Time (s)              0.000XXX        0.00XXXX
Average Confidence             0.96XX          0.97XX
============================================================
```

In [ ]:
unittests.exercise_7(compare_classifiers)

<a name='7'></a>
## 7 - Spam Classification

Finally, let's build a complete spam classifier using Multinomial Naive Bayes. This is one of the most common applications of Naive Bayes in the real world.

<a name='ex-8'></a>
#### **Exercise 8 - `build_spam_classifier`**

**Your Task:**

Implement the `build_spam_classifier` function that creates a complete spam detection pipeline with feature extraction, model training, and interpretable probability scores.

You need to implement:

* **Feature extraction**:
    * Convert text messages into word count features.
    * Use a simple bag-of-words approach or CountVectorizer.
    * Create a vocabulary of the most common words.

* **Train Multinomial Naive Bayes**:
    * Use your MultinomialNB class on the feature matrix.

* **Make predictions with probabilities**:
    * Classify messages as spam or ham (not spam).
    * Return probability scores for interpretability.

* **Identify top spam indicators**:
    * Find which words have the highest probability ratios for spam vs. ham.
    * This helps explain why messages are classified as spam.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**For feature extraction:**
* Use sklearn's CountVectorizer: `from sklearn.feature_extraction.text import CountVectorizer`.
* Fit on training data: `vectorizer = CountVectorizer(max_features=100).fit(X_train)`.
* Transform both train and test: `X_train_counts = vectorizer.transform(X_train).toarray()`.

**For training:**
* Use your MultinomialNB class: `model = MultinomialNB().fit(X_train_counts, y_train)`.

**For spam indicators:**
* Compare feature log probabilities between classes.
* Calculate log ratio: `log_ratio = feature_log_probs[spam] - feature_log_probs[ham]`.
* Sort by ratio to find top spam words: `np.argsort(log_ratio)[-10:]`.
* Map indices back to words using vectorizer: `vectorizer.get_feature_names_out()`.

</details>

In [ ]:
# GRADED FUNCTION: build_spam_classifier
def build_spam_classifier(X_train, X_test, y_train, y_test, max_features=100):
    """
    Build a complete spam classifier with feature extraction and interpretation.

    Args:
        X_train: list of strings OR numeric array - Training messages / features
        X_test: list of strings OR numeric array - Test messages / features
        y_train: numpy array - Training labels (0=ham, 1=spam)
        y_test: numpy array - Test labels
        max_features: int - Maximum number of features (used only for text input)

    Returns:
        results: dict containing:
            - 'model': trained MultinomialNB model
            - 'vectorizer': fitted CountVectorizer (or None for numeric input)
            - 'accuracy': float - Test accuracy
            - 'predictions': numpy array - Predicted labels
            - 'probabilities': numpy array - Prediction probabilities
            - 'top_spam_words': list of (word, score) tuples
    """
    from sklearn.feature_extraction.text import CountVectorizer

    results = {}
    
    ### START CODE HERE ###
    
    # Feature extraction
    
    # Train model
    
    # Make predictions
    
    # Calculate accuracy
    
    # Find top spam indicator words
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Create a toy spam dataset
spam_messages = [
    "Free money now! Click here to win!",
    "Congratulations! You've won a prize. Claim now!",
    "Buy now! Limited time offer! Free shipping!",
    "Win big! Click this link for free money!",
    "Get rich quick! Free trial available now!",
]

ham_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Can you send me the report by end of day?",
    "Thanks for your help with the project.",
    "Let me know when you're available to chat.",
    "I'll send you the documents this afternoon.",
]

# Combine and create labels
X_spam = spam_messages + ham_messages
y_spam = np.array([1]*5 + [0]*5)

# Create train/test split
X_train_spam = X_spam[:8]
X_test_spam = X_spam[8:]
y_train_spam = y_spam[:8]
y_test_spam = y_spam[8:]

# Build spam classifier
spam_results = build_spam_classifier(X_train_spam, X_test_spam, y_train_spam, y_test_spam, max_features=50)

print(f"Spam Classifier Accuracy: {spam_results['accuracy']:.4f}")
print(f"\nTest Predictions: {spam_results['predictions']}")
print(f"True Labels: {y_test_spam}")

print("\nTop Spam Indicator Words:")
for word, score in spam_results['top_spam_words'][:10]:
    print(f"  {word}: {score:.4f}")

# Test on new messages
print("\n" + "="*60)
print("TESTING ON NEW MESSAGES")
print("="*60)

new_messages = [
    "Free money and prizes! Click now!",
    "Can we schedule a meeting next week?",
]

new_features = spam_results['vectorizer'].transform(new_messages).toarray()
new_predictions = spam_results['model'].predict(new_features)
new_probs = spam_results['model'].predict_proba(new_features)

for msg, pred, prob in zip(new_messages, new_predictions, new_probs):
    label = "SPAM" if pred == 1 else "HAM"
    confidence = prob[pred]
    print(f"\nMessage: '{msg}'")
    print(f"Prediction: {label} (confidence: {confidence:.4f})")

##### **Expected Output** (exact words and scores may vary)
```
Spam Classifier Accuracy: 1.0000

Test Predictions: [1 0]
True Labels: [1 0]

Top Spam Indicator Words:
  free: X.XXXX
  money: X.XXXX
  now: X.XXXX
  ...

============================================================
TESTING ON NEW MESSAGES
============================================================

Message: 'Free money and prizes! Click now!'
Prediction: SPAM (confidence: 0.XXXX)

Message: 'Can we schedule a meeting next week?'
Prediction: HAM (confidence: 0.XXXX)
```

In [ ]:
unittests.exercise_8(build_spam_classifier)

## Congratulations!

You have successfully completed the Naive Bayes assignment! You've built a comprehensive understanding of one of the most elegant and interpretable machine learning algorithms.

### What You Accomplished:

1.  **Understood Bayes' Theorem**: Learned the mathematical foundation of probabilistic classification
2.  **Implemented Gaussian Naive Bayes**: Built a classifier for continuous features from scratch
3.  **Calculated Likelihood Probabilities**: Used Gaussian distributions to model feature probabilities
4.  **Made Predictions with Probabilities**: Applied Bayes' theorem to generate interpretable predictions
5.  **Implemented Probability Calibration**: Improved prediction confidence using Platt scaling
6.  **Built Multinomial Naive Bayes**: Created a classifier for discrete features and text data
7.  **Compared Classifiers**: Analyzed when Naive Bayes excels compared to Logistic Regression
8.  **Created a Spam Classifier**: Built a complete real-world application with interpretable results

### Key Takeaways:

* **Naive Bayes is fast and efficient**: Perfect for real-time applications and large datasets
* **Interpretability matters**: Probability scores help explain model decisions
* **Feature independence assumption**: Despite being "naive", it works remarkably well in practice
* **Text classification**: Multinomial Naive Bayes excels at document and spam classification
* **Probability calibration**: Raw probabilities can be improved for better decision-making

**Great work! You now have a solid foundation in probabilistic machine learning!**